# NevoScan
## Эксперименты 2, 3, 4
### Классификация клинических признаков дерматоскопических изображений.

Val и Test только на Derm7pt во всех экспериментах

| Эксп. | Датасет train | Каналы | Маска | Аугментации |
|-------|--------------|--------|-------|-------------|
| **1** | Derm7pt | 3 RGB | нет | flip, rotation |
| **2** | ISIC Task 2 | 3 RGB | нет | flip |
| **3** | ISIC Task 2 | 4 RGB+маска | маски признаков | flip |
| **4** | Derm7pt + ISIC | 4 RGB+маска | маски признаков | flip + CLAHE + updown |



In [ ]:
import torch
print(f'CUDA доступен: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  GPU не найден! Зайди в Runtime → Change runtime type → T4 GPU')


### 1. Скачиваем ISIC 2018 Task 2


In [ ]:
import os
os.makedirs('/content/isic', exist_ok=True)

if not os.path.exists('/content/isic/images'):
    print('Скачиваем изображения ISIC (~5 GB)...')
    !wget -q --show-progress -O /content/isic/images.zip \
        'https://isic-challenge-data.s3.amazonaws.com/2018/ISIC2018_Task1-2_Training_Input.zip'
    !unzip -q /content/isic/images.zip -d /content/isic/tmp
    !mv '/content/isic/tmp/ISIC2018_Task1-2_Training_Input' /content/isic/images
    !rm -rf /content/isic/images.zip /content/isic/tmp
    print('✓ Изображения готовы!')
else:
    print('Изображения уже есть')

if not os.path.exists('/content/isic/masks'):
    print('Скачиваем маски признаков...')
    !wget -q --show-progress -O /content/isic/masks.zip \
        'https://isic-challenge-data.s3.amazonaws.com/2018/ISIC2018_Task2_Training_GroundTruth_v3.zip'
    !unzip -q /content/isic/masks.zip -d /content/isic/tmp
    !mv '/content/isic/tmp/ISIC2018_Task2_Training_GroundTruth_v3' /content/isic/masks
    !rm -rf /content/isic/masks.zip /content/isic/tmp
    print('✓ Маски готовы!')
else:
    print('Маски уже есть')

n_imgs  = len([f for f in os.listdir('/content/isic/images') if f.endswith('.jpg')])
n_masks = len(os.listdir('/content/isic/masks'))
print(f'\nИзображений ISIC: {n_imgs}')
print(f'Файлов масок:     {n_masks}')
print('\nПримеры масок:')
for f in sorted(os.listdir('/content/isic/masks'))[:5]:
    print(f'  {f}')


### 2. Монтируем Derm7pt для val/test и Эксп. 4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/dataset', exist_ok=True)
if not os.path.exists('/content/dataset/release_v0'):
    !unzip -q "/content/drive/MyDrive/Диплом/практика_преддипломная/derm7pt.zip" -d "/content/dataset"
    print('✓ Derm7pt распакован')
else:
    print('Derm7pt уже есть')
!find /content/dataset -name 'meta.csv'


### 3. Импорты и конфиг

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Пути
DERM_BASE  = '/content/dataset/release_v0'
DERM_IMG   = os.path.join(DERM_BASE, 'images')
DERM_META  = os.path.join(DERM_BASE, 'meta/meta.csv')
DERM_TRAIN = os.path.join(DERM_BASE, 'meta/train_indexes.csv')
DERM_VAL   = os.path.join(DERM_BASE, 'meta/valid_indexes.csv')
DERM_TEST  = os.path.join(DERM_BASE, 'meta/test_indexes.csv')
ISIC_IMG   = '/content/isic/images'
ISIC_MASKS = '/content/isic/masks'

# Гиперпараметры
IMG_SIZE     = 300
BATCH_SIZE   = 16
NUM_EPOCHS   = 10
LR           = 1e-4
WEIGHT_DECAY = 1e-3
THRESHOLD    = 0.4
FOCAL_ALPHA  = 0.25
FOCAL_GAMMA  = 2.0

# Признаки по шкале Аргензиано
FEATURES = [
    'pigment_network', 'streaks', 'pigmentation',
    'regression_structures', 'dots_and_globules',
    'blue_whitish_veil', 'vascular_structures'
]
FEAT_RU = [
    'Пигм. сеть', 'Полосы', 'Пигментация',
    'Регрессия', 'Точки/Глобулы', 'Бело-гол. вуаль', 'Сос. структуры'
]

# ISIC Task 2 содержит: pigment_network, negative_network, streaks, milia_like_cysts, globules
ISIC_ATTR = {
    'pigment_network':       'pigment_network',
    'streaks':               'streaks',
    'dots_and_globules':     'globules',
    'pigmentation':          None,   # нет в ISIC → label=-1
    'regression_structures': None,
    'blue_whitish_veil':     None,
    'vascular_structures':   None,
}
print('✓ Конфиг загружен')


### 4. Загружаем Derm7pt

In [ ]:
def map_label(text):
    if pd.isna(text): return 0
    return 0 if str(text).lower().strip() in ['absent','regular','typical'] else 1

df_meta = pd.read_csv(DERM_META)
df_derm = df_meta.copy()
for feat in FEATURES:
    df_derm[feat] = df_derm[feat].apply(map_label)

all_files = glob.glob(os.path.join(DERM_IMG,'**/*'), recursive=True)
path_map  = {f.lower(): f for f in all_files if os.path.isfile(f)}
def find_derm(fname):
    return path_map.get(os.path.join(DERM_IMG, fname).lower())

df_derm['full_path'] = df_derm['derm'].apply(find_derm)
df_derm['mask_path'] = None
df_derm['source']    = 'derm7pt'
df_derm = df_derm[df_derm['full_path'].notna()].reset_index(drop=True)
print(f'Derm7pt: {len(df_derm)} изображений')

train_idx = pd.read_csv(DERM_TRAIN)['indexes'].values
val_idx   = pd.read_csv(DERM_VAL)['indexes'].values
test_idx  = pd.read_csv(DERM_TEST)['indexes'].values

derm_train = df_derm.iloc[train_idx].reset_index(drop=True)
derm_val   = df_derm.iloc[val_idx].reset_index(drop=True)
derm_test  = df_derm.iloc[test_idx].reset_index(drop=True)
print(f'Train: {len(derm_train)} | Val: {len(derm_val)} | Test: {len(derm_test)}')


### 5. EDA: анализ Derm7pt (Val/Test выборка)

Смотрим распределение меток, нам это важно для понимания baseline метрик.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# График 1: доля положительных меток в каждом сплите
splits = {'Train': derm_train, 'Val': derm_val, 'Test': derm_test}
pos_rates = {}
for split_name, split_df in splits.items():
    pos_rates[split_name] = [split_df[f].mean()*100 for f in FEATURES]

x = np.arange(len(FEATURES))
width = 0.25
colors = ['#4C72B0', '#55A868', '#C44E52']
for i, (name, rates) in enumerate(pos_rates.items()):
    axes[0].bar(x + i*width, rates, width, label=name, color=colors[i], alpha=0.85)
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(FEAT_RU, rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('Доля положительных меток, %')
axes[0].set_title('Распределение меток по сплитам (Derm7pt)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# График 2: матрица корреляции признаков (val)
corr = derm_val[FEATURES].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            xticklabels=FEAT_RU, yticklabels=FEAT_RU,
            ax=axes[1], annot_kws={'size': 8})
axes[1].set_title('Корреляция между признаками (Val)')
axes[1].tick_params(axis='x', rotation=30, labelsize=8)
axes[1].tick_params(axis='y', rotation=0, labelsize=8)

plt.tight_layout()
plt.savefig('/content/eda_derm7pt.png', dpi=150, bbox_inches='tight')
plt.show()

# Таблица дисбаланса
print('\nТаблица дисбаланса меток (Val = 203 изображения):')
print(f'{"Признак":<28} {"Pos":>5} {"Neg":>5} {"Pos%":>7}')
print('-'*48)
for feat, name in zip(FEATURES, FEAT_RU):
    pos = int(derm_val[feat].sum())
    neg = len(derm_val) - pos
    print(f'{name:<28} {pos:>5} {neg:>5} {100*pos/len(derm_val):>6.1f}%')


### 6. Парсим ISIC Task 2

In [ ]:
os.makedirs('/content/isic/combined_masks', exist_ok=True)

def get_isic_label(img_id, feat):
    attr = ISIC_ATTR.get(feat)
    if attr is None: return -1
    mf = os.path.join(ISIC_MASKS, f'{img_id}_attribute_{attr}.png')
    if not os.path.exists(mf): return 0
    return 1 if np.array(Image.open(mf).convert('L')).max() > 0 else 0

def build_combined_mask(img_id):
    combined = None
    for attr in ['pigment_network','negative_network','streaks','milia_like_cysts','globules']:
        mf = os.path.join(ISIC_MASKS, f'{img_id}_attribute_{attr}.png')
        if os.path.exists(mf):
            m = np.array(Image.open(mf).convert('L'))
            combined = m if combined is None else np.maximum(combined, m)
    if combined is None: return None
    out = f'/content/isic/combined_masks/{img_id}_combined.png'
    Image.fromarray(combined).save(out)
    return out

isic_files = sorted([f for f in os.listdir(ISIC_IMG) if f.endswith('.jpg')])
print(f'Изображений ISIC: {len(isic_files)}')

records = []
for fname in tqdm(isic_files, desc='Парсим ISIC'):
    img_id = os.path.splitext(fname)[0]
    row = {'full_path': os.path.join(ISIC_IMG, fname), 'source': 'isic'}
    for feat in FEATURES:
        row[feat] = get_isic_label(img_id, feat)
    row['mask_path'] = build_combined_mask(img_id)
    records.append(row)

isic_df = pd.DataFrame(records)
print(f'\nISIC: {len(isic_df)} записей | Масок: {isic_df["mask_path"].notna().sum()}')


### 7. EDA: анализ ISIC Task 2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# График 1: распределение аннотированных меток ISIC
feat_data = []
for feat, name in zip(FEATURES, FEAT_RU):
    ann = (isic_df[feat] >= 0).sum()
    pos = (isic_df[feat] == 1).sum()
    neg = (isic_df[feat] == 0).sum()
    na  = (isic_df[feat] == -1).sum()
    feat_data.append({'name': name, 'pos': pos, 'neg': neg, 'na': na, 'ann': ann})

x = np.arange(len(FEAT_RU))
pos_vals = [d['pos'] for d in feat_data]
neg_vals = [d['neg'] for d in feat_data]
na_vals  = [d['na']  for d in feat_data]
axes[0].bar(x, pos_vals, label='Позитив (1)', color='#C44E52', alpha=0.85)
axes[0].bar(x, neg_vals, bottom=pos_vals, label='Негатив (0)', color='#4C72B0', alpha=0.85)
axes[0].bar(x, na_vals, bottom=[p+n for p,n in zip(pos_vals,neg_vals)],
            label='Не аннотировано (-1)', color='#cccccc', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(FEAT_RU, rotation=30, ha='right', fontsize=9)
axes[0].set_title('Распределение меток в ISIC Task 2')
axes[0].set_ylabel('Количество изображений')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

# График 2: сравнение % положительных: Derm7pt val vs ISIC (только аннотированные)
derm_pos = [derm_val[f].mean()*100 for f in FEATURES]
isic_pos = []
for feat in FEATURES:
    ann = (isic_df[feat] >= 0).sum()
    pos = (isic_df[feat] == 1).sum()
    isic_pos.append(100*pos/ann if ann > 0 else 0)

w = 0.35
axes[1].bar(x - w/2, derm_pos, w, label='Derm7pt Val', color='#4C72B0', alpha=0.85)
axes[1].bar(x + w/2, isic_pos, w, label='ISIC Task 2', color='#55A868', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(FEAT_RU, rotation=30, ha='right', fontsize=9)
axes[1].set_title('Сравнение % положительных: Derm7pt vs ISIC')
axes[1].set_ylabel('Доля положительных, %')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/eda_isic.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nСтатистика ISIC Task 2:')
print(f'{"Признак":<28} {"Аннот.":>7} {"Pos":>6} {"Pos%":>7} {"Маска":>8}')
print('-'*58)
for feat, name in zip(FEATURES, FEAT_RU):
    ann = int((isic_df[feat] >= 0).sum())
    pos = int((isic_df[feat] == 1).sum())
    pct = 100*pos/ann if ann > 0 else 0
    has_mask = 'есть' if ISIC_ATTR.get(feat) else 'нет (-1)'
    print(f'{name:<28} {ann:>7} {pos:>6} {pct:>6.1f}% {has_mask:>8}')


### 8. Формируем сплиты для экспериментов

In [ ]:
val_df  = derm_val.copy() # Val только Derm7pt (одинаково для всех экспериментов)
test_df = derm_test.copy() # Test только Derm7pt

train_exp2 = isic_df.copy() # Эксп.2: ISIC only, 3ch
train_exp3 = isic_df.copy() # Эксп.3: ISIC only, 4ch + маска
train_exp4 = pd.concat([isic_df, derm_train], ignore_index=True) # Эксп.4: ISIC+Derm, 4ch + маска + aug

print('Сплиты сформированы:')
print(f'  Val  (все эксп.): {len(val_df):>5} изображений | Derm7pt')
print(f'  Test (все эксп.): {len(test_df):>5} изображений | Derm7pt')
print(f'  Train Эксп.2:     {len(train_exp2):>5} изображений | ISIC only')
print(f'  Train Эксп.3:     {len(train_exp3):>5} изображений | ISIC only + маски')
print(f'  Train Эксп.4:     {len(train_exp4):>5} изображений | ISIC={( train_exp4["source"]=="isic").sum()} + Derm7pt={(train_exp4["source"]=="derm7pt").sum()}')


### 9. Все классы: датасеты, модели, лосс, обучение, визуализация

In [ ]:
# Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__(); self.a, self.g = alpha, gamma
    def forward(self, inp, tgt):
        bce = nn.functional.binary_cross_entropy_with_logits(inp, tgt, reduction='none')
        return (self.a*(1-torch.exp(-bce))**self.g*bce).mean()

class MaskedFocalLoss(nn.Module):
    """
    Focal Loss с маскированием меток -1 и усиленным штрафом
    за ложноотрицательные предсказания (false negative).
    fn_weight > 1 увеличивает стоимость пропуска патологии.
    """
    def __init__(self, alpha=0.25, gamma=2.0, fn_weight=3.0):
        super().__init__()
        self.a, self.g, self.fn_w = alpha, gamma, fn_weight

    def forward(self, inp, tgt):
        mask   = (tgt >= 0).float()
        tgt_c  = tgt.clamp(min=0)
        # pos_weight: для позитивных примеров умножаем лосс на fn_weight
        pw     = torch.ones_like(inp) + (self.fn_w - 1) * tgt_c
        bce    = nn.functional.binary_cross_entropy_with_logits(
                     inp, tgt_c, weight=pw, reduction='none')
        fl     = self.a * (1 - torch.exp(-bce)) ** self.g * bce
        n      = mask.sum()
        return (fl * mask).sum() / n if n > 0 else fl.sum() * 0


# Датасет_Эксп. 2 _ 3 канала
class SkinDataset3Ch(Dataset):
    def __init__(self, df, is_train=False, sz=300):
        self.df = df.reset_index(drop=True)
        self.is_train, self.sz = is_train, sz
        self.mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        self.std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['full_path']).convert('RGB').resize((self.sz,self.sz), Image.BILINEAR)
        t   = torch.from_numpy(np.array(img,np.float32)/255.).permute(2,0,1)
        t   = (t-self.mean)/self.std
        if self.is_train:
            if torch.rand(1)>.5: t=torch.flip(t,[2])
            if torch.rand(1)>.5: t=torch.flip(t,[1])
        return t, torch.tensor(row[FEATURES].values.astype(np.float32))

# Датасет_Эксп. 3 и 4 _ 4 канала
class SkinDataset4Ch(Dataset):
    def __init__(self, df, is_train=False, sz=300, use_clahe=False, use_quality_aug=False):
        self.df = df.reset_index(drop=True)
        self.is_train, self.sz = is_train, sz
        self.use_clahe = use_clahe
        self.use_qa    = use_quality_aug
        self.mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        self.std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    def __len__(self): return len(self.df)

    def _clahe(self, arr):
        try:
            import cv2
            u8=( arr*255).astype(np.uint8)
            lab=cv2.cvtColor(u8,cv2.COLOR_RGB2LAB)
            cl=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
            lab[:,:,0]=cl.apply(lab[:,:,0])
            return cv2.cvtColor(lab,cv2.COLOR_LAB2RGB).astype(np.float32)/255.
        except: return arr

    def _quality_aug(self, img):
        if torch.rand(1)>0.3: return img
        scale=torch.FloatTensor(1).uniform_(0.5,0.9).item()
        small=max(64,int(self.sz*scale))
        return img.resize((small,small),Image.BILINEAR).resize((self.sz,self.sz),Image.BILINEAR)

    def _load_mask(self, mask_path):
        if pd.notna(mask_path) and mask_path and os.path.exists(str(mask_path)):
            return np.array(Image.open(mask_path).convert('L').resize(
                (self.sz,self.sz),Image.NEAREST),np.float32)/255.
        return np.zeros((self.sz,self.sz),np.float32)

    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=Image.open(row['full_path']).convert('RGB')
        if self.is_train and self.use_qa: img=self._quality_aug(img)
        img=img.resize((self.sz,self.sz),Image.BILINEAR)
        arr=np.array(img,np.float32)/255.
        if self.is_train and self.use_clahe and torch.rand(1)>0.5: arr=self._clahe(arr)
        mask=self._load_mask(row.get('mask_path'))
        t4=torch.from_numpy(np.concatenate([arr,mask[:,:,np.newaxis]],axis=2)).permute(2,0,1)
        t4[:3]=(t4[:3]-self.mean)/self.std
        t4[3]=(t4[3]-0.5)/0.5
        if self.is_train:
            if torch.rand(1)>.5: t4=torch.flip(t4,[2])
            if torch.rand(1)>.5: t4=torch.flip(t4,[1])
        return t4, torch.tensor(row[FEATURES].values.astype(np.float32))

# Модели
def build_3ch():
    m=timm.create_model('efficientnet_b3',pretrained=True,num_classes=len(FEATURES))
    print(f'[3ch] {sum(p.numel() for p in m.parameters())/1e6:.1f}M params')
    return m

def build_4ch():
    m=timm.create_model('efficientnet_b3',pretrained=True,num_classes=len(FEATURES))
    old=m.conv_stem
    new=nn.Conv2d(4,old.out_channels,old.kernel_size,old.stride,old.padding,bias=old.bias is not None)
    with torch.no_grad():
        new.weight[:,:3]=old.weight
        new.weight[:,3:]=old.weight.mean(dim=1,keepdim=True)
    m.conv_stem=new
    print(f'[4ch] 3→4 каналов, ImageNet веса сохранены')
    return m

# Train/eval
def train_epoch(model, loader, opt, crit):
    model.train(); total=0
    for x,y in tqdm(loader, desc='train', leave=False):
        x,y=x.to(device),y.to(device)
        opt.zero_grad()
        loss=crit(model(x),y)
        loss.backward(); opt.step()
        total+=loss.item()
    return total/len(loader)

@torch.no_grad()
def evaluate(model, loader, crit):
    model.eval()
    total,all_pr,all_pd,all_lb=0,[],[],[]
    for x,y in tqdm(loader, desc='eval ', leave=False):
        x,y=x.to(device),y.to(device)
        out=model(x)
        total+=crit(out,y).item()
        pr=torch.sigmoid(out).cpu().numpy()
        all_pr.append(pr)
        all_pd.append((pr>=THRESHOLD).astype(int))
        all_lb.append(y.cpu().numpy())
    probs=np.vstack(all_pr); preds=np.vstack(all_pd); lbls=np.vstack(all_lb)
    f1s,aucs=[],[]
    for i in range(lbls.shape[1]):
        m=lbls[:,i]>=0
        if not m.any(): continue
        f1s.append(f1_score(lbls[m,i],preds[m,i],zero_division=0))
        try: aucs.append(roc_auc_score(lbls[m,i],probs[m,i]))
        except: aucs.append(0.)
    return {'loss':total/len(loader),'macro_f1':float(np.mean(f1s)),
            'macro_auc':float(np.mean(aucs)),'preds':preds,'labels':lbls,'probs':probs}

# цикл обучения
def run_experiment(name, model, train_ds, val_ds, criterion, save_path):
    model=model.to(device)
    ld_tr=DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, drop_last=True)
    ld_vl=DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    x,_=next(iter(ld_tr))
    print(f'Форма тензора: {tuple(x.shape)}')
    opt  =optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sched=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=NUM_EPOCHS)
    best=float('inf'); tl,vl,vf=[],[],[]
    for ep in range(1,NUM_EPOCHS+1):
        tr=train_epoch(model,ld_tr,opt,criterion)
        vm=evaluate(model,ld_vl,criterion)
        sched.step()
        tl.append(tr); vl.append(vm['loss']); vf.append(vm['macro_f1'])
        print(f'[{name}] ep {ep:02d}/{NUM_EPOCHS} | '
              f'train={tr:.4f}  val={vm["loss"]:.4f}  '
              f'F1={vm["macro_f1"]:.3f}  AUC={vm["macro_auc"]:.3f}')
        if vm['loss']<best:
            best=vm['loss']; torch.save(model.state_dict(),save_path)
            print(f'  ✓ best сохранён (ep {ep})')
    model.load_state_dict(torch.load(save_path))
    final=evaluate(model,ld_vl,criterion)
    plot_curves(name, tl, vl, vf, save_path)
    return model, final

# Визуализация
def plot_curves(name, tl, vl, vf, save_path):
    fig, axes = plt.subplots(1,2,figsize=(14,4))
    fig.suptitle(f'Кривые обучения — {name}', fontsize=13)
    axes[0].plot(tl, label='Train Loss', marker='o', ms=4, color='#4C72B0')
    axes[0].plot(vl, label='Val Loss',   marker='s', ms=4, ls='--', color='#C44E52')
    # Отмечаем лучшую эпоху
    best_ep = int(np.argmin(vl))
    axes[0].axvline(best_ep, color='gray', ls=':', alpha=0.7, label=f'Best ep {best_ep+1}')
    axes[0].set_xlabel('Эпоха'); axes[0].set_title('Focal Loss'); axes[0].legend(); axes[0].grid(alpha=.3)
    axes[1].plot(vf, color='#55A868', marker='o', ms=4, label='Macro F1')
    axes[1].axhline(0.5, ls='--', color='red', alpha=.6, label='Baseline 0.5')
    axes[1].axhline(0.499, ls=':', color='orange', alpha=.6, label='Эксп.1 (0.499)')
    axes[1].set_xlabel('Эпоха'); axes[1].set_title('Macro F1 на валидации')
    axes[1].legend(); axes[1].grid(alpha=.3)
    plt.tight_layout()
    fname = save_path.replace('.pth','_curves.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight'); plt.show()

def plot_f1_bars(metrics_dict):
    """Grouped bar chart F1 и AUC по признакам (как в практике)"""
    n_exp = len(metrics_dict)
    x = np.arange(len(FEAT_RU))
    width = 0.8 / n_exp
    colors = ['#4C72B0','#55A868','#C44E52','#DD8452']

    fig, axes = plt.subplots(1,2,figsize=(16,5))
    for i, (exp_name, m) in enumerate(metrics_dict.items()):
        f1s, aucs = [], []
        for j in range(len(FEATURES)):
            mask = m['labels'][:,j] >= 0
            if mask.any():
                f1s.append(f1_score(m['labels'][mask,j], m['preds'][mask,j], zero_division=0))
                try: aucs.append(roc_auc_score(m['labels'][mask,j], m['probs'][mask,j]))
                except: aucs.append(0.)
            else:
                f1s.append(0.); aucs.append(0.)
        offset = (i - n_exp/2 + 0.5) * width
        axes[0].bar(x+offset, f1s,  width-0.02, label=exp_name, color=colors[i], alpha=0.85)
        axes[1].bar(x+offset, aucs, width-0.02, label=exp_name, color=colors[i], alpha=0.85)

    for ax, title, baseline in [(axes[0],'F1-score по признакам',0.5),
                                 (axes[1],'AUC-ROC по признакам',0.5)]:
        ax.axhline(baseline, ls='--', color='red', alpha=0.5, label='0.5 baseline')
        ax.set_xticks(x); ax.set_xticklabels(FEAT_RU, rotation=30, ha='right', fontsize=9)
        ax.set_ylim(0,1); ax.set_title(title); ax.legend(fontsize=8); ax.grid(axis='y',alpha=.3)
    plt.tight_layout()
    plt.savefig('/content/comparison_f1_auc.png', dpi=150, bbox_inches='tight'); plt.show()

def plot_confusion_matrices(m, exp_name):
    """Confusion matrix для каждого из 7 признаков (как в практике)"""
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    fig.suptitle(f'Confusion Matrices — {exp_name} (Val = Derm7pt)', fontsize=13)
    axes_flat = axes.flatten()
    for i, (feat, name) in enumerate(zip(FEATURES, FEAT_RU)):
        ax = axes_flat[i]
        mask = m['labels'][:,i] >= 0
        if not mask.any():
            ax.set_visible(False); continue
        cm = confusion_matrix(m['labels'][mask,i], m['preds'][mask,i])
        f1  = f1_score(m['labels'][mask,i], m['preds'][mask,i], zero_division=0)
        rec = recall_score(m['labels'][mask,i], m['preds'][mask,i], zero_division=0)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Норма','Патология'],
                    yticklabels=['Норма','Патология'], cbar=False)
        ax.set_title(f'{name}\nF1={f1:.2f}  Recall={rec:.2f}', fontsize=10)
        ax.set_xlabel('Предсказание'); ax.set_ylabel('Истина')
    axes_flat[-1].set_visible(False)
    plt.tight_layout()
    fname = f'/content/confusion_{exp_name.replace(" ","_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight'); plt.show()

def print_metrics_table(label, m):
    print(f'\n{"="*62}')
    print(f'  {label}')
    print(f'  Macro F1: {m["macro_f1"]:.4f}  |  Macro AUC: {m["macro_auc"]:.4f}')
    print(f'{"─"*62}')
    print(f'{"Признак":<26} {"F1":>6} {"Prec":>6} {"Recall":>7} {"AUC":>7} {"Pos":>5}')
    print(f'{"─"*62}')
    for i, name in enumerate(FEAT_RU):
        msk = m['labels'][:,i] >= 0
        if not msk.any(): print(f'{name:<26} не аннотировано'); continue
        f1  = f1_score(m['labels'][msk,i],  m['preds'][msk,i],  zero_division=0)
        pr  = precision_score(m['labels'][msk,i], m['preds'][msk,i], zero_division=0)
        rec = recall_score(m['labels'][msk,i],    m['preds'][msk,i], zero_division=0)
        try: auc=roc_auc_score(m['labels'][msk,i], m['probs'][msk,i])
        except: auc=0.
        pos = int(m['labels'][msk,i].sum())
        print(f'{name:<26} {f1:>6.3f} {pr:>6.3f} {rec:>7.3f} {auc:>7.3f} {pos:>5}')
    print(f'{"="*62}')

print('✓ Все утилиты загружены')


### 10. Эксперименты

## Эксперимент 2

**Датасет:** ISIC Task 2

**Цель:** оценить насколько большой датасет ISIC улучшает качество на Derm7pt.

**Архитектура:** EfficientNet-B3, 3 канала RGB, Masked Focal Loss

**Train:** ISIC Task 2 (около 2600 изображений)

**Val/Test:** Derm7pt


NUM_EPOCHS = 15 (потом исправили на 10 для эксп 3 и 4)

WEIGHT_DECAY = 1e-2 (потом исправили на 1e-3 для эксп 3 и 4)

In [ ]:
print('='*62)
print('ЭКСПЕРИМЕНТ 2: ISIC Task 2, 3ch, без маски')
print('='*62)

model2, metrics2 = run_experiment(
    name      = 'Эксп.2 ISIC 3ch',
    model     = build_3ch(),
    train_ds  = SkinDataset3Ch(train_exp2, is_train=True),
    val_ds    = SkinDataset3Ch(val_df,     is_train=False),
    criterion = MaskedFocalLoss(alpha=0.25, gamma=2.0, fn_weight=3.0),
    save_path = '/content/best_exp2.pth',
)
print_metrics_table('ЭКСП. 2  |  Val = Derm7pt', metrics2)
plot_confusion_matrices(metrics2, 'Эксп.2 ISIC 3ch')


## Эксперимент 3

**Датасет:** ISIC Task 2

**Цель:** проверить даёт ли добавление маски признаков прирост качества.

**Архитектура:** EfficientNet-B3, 4 канала (RGB + маска)

**4-й канал:** сводная маска (OR всех атрибутных масок ISIC).  
Для Val (Derm7pt): нулевой канал (честный ablation), т.к. масок в Derm7pt нет.


In [ ]:
print('='*62)
print('ЭКСПЕРИМЕНТ 3: ISIC Task 2, 4ch, с маской')
print('='*62)

model3, metrics3 = run_experiment(
    name      = 'Эксп.3 ISIC 4ch+маска',
    model     = build_4ch(),
    train_ds  = SkinDataset4Ch(train_exp3, is_train=True,
                               use_clahe=False, use_quality_aug=False),
    val_ds    = SkinDataset4Ch(val_df,     is_train=False),
    criterion = MaskedFocalLoss(alpha=0.25, gamma=2.0, fn_weight=3.0),
    save_path = '/content/best_exp3.pth',
)
print_metrics_table('ЭКСП. 3  |  Val = Derm7pt', metrics3)
plot_confusion_matrices(metrics3, 'Эксп.3 ISIC 4ch')


## Эксперимент 4

**Датасет:** Derm7pt + ISIC Task 2

**Цель:** расширить датасет, добавить маски и проверить даёт ли это улучшение качества.

**Архитектура:** EfficientNet-B3, 4 канала (RGB + маска)

**Новые аугментации:**
- CLAHE для усиления контраста, чтобы были лучше видны слабые признаки (Бело-голубая вуаль, Регрессия)
- Upscaling/downscaling для имитации разного качества дерматоскопов


In [ ]:
print('='*62)
print('ЭКСПЕРИМЕНТ 4: Derm7pt + ISIC, 4ch + CLAHE + updown')
print('='*62)

model4, metrics4 = run_experiment(
    name      = 'Эксп.4 Derm+ISIC 4ch+aug',
    model     = build_4ch(),
    train_ds  = SkinDataset4Ch(train_exp4, is_train=True,
                               use_clahe=True, use_quality_aug=True),
    val_ds    = SkinDataset4Ch(val_df,     is_train=False),
    criterion = MaskedFocalLoss(alpha=0.25, gamma=2.0, fn_weight=3.0),
    save_path = '/content/best_exp4.pth',
)
print_metrics_table('ЭКСП. 4  |  Val = Derm7pt', metrics4)
plot_confusion_matrices(metrics4, 'Эксп.4 Derm+ISIC 4ch')


### 11. Сводный анализ всех экспериментов

Сравниваем F1 и AUC-ROC по каждому признаку

In [ ]:
# Результаты Эксп.1 из преддипломной практики
# Создаём псевдо-объект с метриками для графика
exp1_f1_per_feat  = [0.346, 0.417, 0.638, 0.358, 0.657, 0.450, 0.364]  # B3 из практики
exp1_auc_per_feat = [0.674, 0.691, 0.629, 0.643, 0.692, 0.827, 0.764]  # B3 из практики

# Собираем метрики по признакам для каждого эксперимента
def get_per_feat_metrics(m):
    f1s, aucs = [], []
    for i in range(len(FEATURES)):
        msk = m['labels'][:,i] >= 0
        if msk.any():
            f1s.append(f1_score(m['labels'][msk,i], m['preds'][msk,i], zero_division=0))
            try: aucs.append(roc_auc_score(m['labels'][msk,i], m['probs'][msk,i]))
            except: aucs.append(0.)
        else:
            f1s.append(0.); aucs.append(0.)
    return f1s, aucs

f1_2,  auc_2  = get_per_feat_metrics(metrics2)
f1_3,  auc_3  = get_per_feat_metrics(metrics3)
f1_4,  auc_4  = get_per_feat_metrics(metrics4)

# Grouped bar chart
x = np.arange(len(FEAT_RU))
w = 0.19
colors = ['#4C72B0','#55A868','#C44E52','#DD8452']
labels_exp = ['Эксп.1 (Derm7pt 3ch)','Эксп.2 (ISIC 3ch)',
              'Эксп.3 (ISIC 4ch+маска)','Эксп.4 (Derm+ISIC 4ch+aug)']

fig, axes = plt.subplots(1,2,figsize=(18,6))
fig.suptitle('Сравнение экспериментов по признакам (Val = Derm7pt)', fontsize=13)

for i, (vals, label) in enumerate(zip(
        [exp1_f1_per_feat, f1_2, f1_3, f1_4], labels_exp)):
    off = (i-1.5)*w
    axes[0].bar(x+off, vals, w-0.02, label=label, color=colors[i], alpha=0.85)
    for j, v in enumerate(vals):
        axes[0].text(x[j]+off, v+0.01, f'{v:.2f}', ha='center', va='bottom',
                     fontsize=6.5, rotation=90)

for i, (vals, label) in enumerate(zip(
        [exp1_auc_per_feat, auc_2, auc_3, auc_4], labels_exp)):
    off = (i-1.5)*w
    axes[1].bar(x+off, vals, w-0.02, label=label, color=colors[i], alpha=0.85)
    for j, v in enumerate(vals):
        axes[1].text(x[j]+off, v+0.01, f'{v:.2f}', ha='center', va='bottom',
                     fontsize=6.5, rotation=90)

for ax, title in [(axes[0],'F1-score по признакам'),(axes[1],'AUC-ROC по признакам')]:
    ax.axhline(0.5, ls='--', color='gray', alpha=0.5, label='0.5')
    ax.set_xticks(x); ax.set_xticklabels(FEAT_RU, rotation=30, ha='right', fontsize=9)
    ax.set_ylim(0, 1.05); ax.set_title(title); ax.legend(fontsize=8); ax.grid(axis='y',alpha=.3)

plt.tight_layout()
plt.savefig('/content/all_experiments_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


### 12. Сводная таблица

In [ ]:
exp1 = {'macro_f1': 0.499, 'macro_auc': 0.703}

all_res = [
    ('Эксп.1  Derm7pt, 3ch, без маски (baseline)', exp1),
    ('Эксп.2  ISIC Task 2, 3ch, без маски',        metrics2),
    ('Эксп.3  ISIC Task 2, 4ch, с маской',          metrics3),
    ('Эксп.4  ISIC+Derm, 4ch, маска+CLAHE+updown',  metrics4),
]

print('\n' + '='*68)
print('  СВОДНАЯ ТАБЛИЦА')
print('  Val = Derm7pt (203 изображения), threshold = 0.4')
print('='*68)
print(f'  {"Конфигурация":<46} {"F1":>6} {"AUC":>7} {"ΔF1":>7}')
print('  ' + '-'*65)
for name, m in all_res:
    delta = m['macro_f1'] - exp1['macro_f1']
    sign  = '+' if delta >= 0 else ''
    print(f'  {name:<46} {m["macro_f1"]:>6.3f} {m["macro_auc"]:>7.3f} {sign}{delta:>6.3f}')
print('='*68)

# Сохраняем результаты
summary = {}
for name, m in all_res:
    summary[name] = {
        'macro_f1':  round(m['macro_f1'], 4),
        'macro_auc': round(m['macro_auc'], 4),
        'delta_f1':  round(m['macro_f1'] - exp1['macro_f1'], 4)
    }
with open('/content/results_summary.json','w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('\n✓ Сохранено в /content/results_summary.json')


### 13. Подсчёт признаков

In [ ]:
# Подсчёт баллов по шкале Argenziano (7-point checklist)
# Большие критерии (2 балла): пигм. сеть, бело-голубая вуаль, сосудистые структуры
# Малые критерии (1 балл): полосы, пигментация, регрессия, точки/глобулы
ARGENZIANO_WEIGHTS = {
    'pigment_network':       2,
    'streaks':               1,
    'pigmentation':          1,
    'regression_structures': 1,
    'dots_and_globules':     1,
    'blue_whitish_veil':     2,
    'vascular_structures':   2,
}
SUSPICION_THRESHOLD = 3 # если сумма ≥ 3, то подозрение на меланому

def compute_argenziano_score(probs_row, threshold=THRESHOLD):
    """
    Вычисляет балл по шкале Argenziano для одного изображения.
    probs_row: массив вероятностей длиной 7 (порядок = FEATURES)
    Возвращает: (score, pred_binary, found_features)
    """
    weights = [ARGENZIANO_WEIGHTS[f] for f in FEATURES]
    preds   = (probs_row >= threshold).astype(int)
    score   = int(np.dot(preds, weights))
    found   = [FEAT_RU[i] for i, p in enumerate(preds) if p == 1]
    return score, int(score >= SUSPICION_THRESHOLD), found


def evaluate_argenziano(metrics, split_name='Val', threshold=THRESHOLD):
    """
    Полный анализ по шкале Argenziano для всего датасета.
    metrics: словарь с 'probs' и 'labels' (из функции evaluate())
    """
    probs  = metrics['probs']   # (N, 7)
    labels = metrics['labels']  # (N, 7)
    N = len(probs)

    scores, verdicts, found_list = [], [], []
    for i in range(N):
        score, verdict, found = compute_argenziano_score(probs[i], threshold)
        scores.append(score)
        verdicts.append(verdict)
        found_list.append(found)

    scores   = np.array(scores)
    verdicts = np.array(verdicts)

    # Таблица: распределение баллов
    print(f'\n{"="*55}')
    print(f'  Анализ по шкале Argenziano — {split_name}')
    print(f'  Порог классификации: {threshold} | Порог подозрения: ≥{SUSPICION_THRESHOLD} баллов')
    print(f'{"="*55}')
    print(f'\n  Распределение суммарных баллов:')
    for s in range(0, 11):
        count = (scores == s).sum()
        if count == 0:
            continue
        bar = '█' * int(count / N * 40)
        flag = '  Подозрение' if s >= SUSPICION_THRESHOLD else ''
        print(f'  {s:2d} баллов: {count:4d} ({100*count/N:5.1f}%)  {bar}{flag}')

    n_susp = verdicts.sum()
    n_norm = N - n_susp
    print(f'\n  Итого:')
    print(f'  Подозрительных (≥{SUSPICION_THRESHOLD} балла): {n_susp} ({100*n_susp/N:.1f}%)')
    print(f'  Норма          (<{SUSPICION_THRESHOLD} балла): {n_norm} ({100*n_norm/N:.1f}%)')

    # Если есть GT диагноз (колонка diagnosis в Derm7pt)
    # Argenziano score vs реальное число позитивных признаков
    gt_sum = (labels >= 0) * labels # исключаем -1
    gt_sum = gt_sum.clip(min=0)
    gt_scores_weighted = np.array([
        sum(ARGENZIANO_WEIGHTS[f] * int(labels[i, j] == 1)
            for j, f in enumerate(FEATURES) if labels[i, j] >= 0)
        for i in range(N)
    ])
    gt_verdicts = (gt_scores_weighted >= SUSPICION_THRESHOLD).astype(int)

    # Confusion matrix: наш вердикт vs GT вердикт по Argenziano
    from sklearn.metrics import confusion_matrix, classification_report
    cm = confusion_matrix(gt_verdicts, verdicts)
    print(f'\n  Confusion matrix (наш вердикт vs GT по Argenziano):')
    print(f'  {"":20s} Pred: Норма  Pred: Подозрение')
    labels_cm = ['GT: Норма', 'GT: Подозрение']
    for row_label, row in zip(labels_cm, cm):
        print(f'  {row_label:20s}   {row[0]:6d}         {row[1]:6d}')

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv  = tp / (tp + fp) if (tp + fp) > 0 else 0
        print(f'\n  Чувствительность (Recall): {sens:.3f}')
        print(f'  Специфичность:             {spec:.3f}')
        print(f'  Точность (Precision):      {ppv:.3f}')

    print(f'{"="*55}')
    return scores, verdicts, gt_verdicts


def plot_argenziano_scores(all_scores_dict):
    """
    Сравнительный график распределения баллов по экспериментам.
    all_scores_dict: {'Эксп.1': scores_array, 'Эксп.4': scores_array, ...}
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('Анализ по шкале Argenziano (Val = Derm7pt)', fontsize=13)

    colors = ['#4C72B0', '#55A868', '#C44E52', '#DD8452']
    bins   = np.arange(-0.5, 11.5, 1)

    # График 1: гистограммы баллов
    for i, (name, scores) in enumerate(all_scores_dict.items()):
        axes[0].hist(scores, bins=bins, alpha=0.6, label=name,
                     color=colors[i % len(colors)], density=True)
    axes[0].axvline(SUSPICION_THRESHOLD - 0.5, color='red', ls='--',
                    lw=2, label=f'Порог подозрения ({SUSPICION_THRESHOLD})')
    axes[0].set_xlabel('Балл по шкале Argenziano')
    axes[0].set_ylabel('Плотность')
    axes[0].set_title('Распределение баллов')
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3)
    axes[0].set_xticks(range(0, 11))

    # График 2: % подозрительных по экспериментам
    exp_names = list(all_scores_dict.keys())
    pct_susp  = [100 * (s >= SUSPICION_THRESHOLD).mean()
                 for s in all_scores_dict.values()]
    bars = axes[1].bar(exp_names, pct_susp,
                       color=colors[:len(exp_names)], alpha=0.85)
    for bar, pct in zip(bars, pct_susp):
        axes[1].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.5,
                     f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)
    axes[1].set_ylabel('% изображений с подозрением на меланому')
    axes[1].set_title('Доля "подозрительных" по экспериментам')
    axes[1].set_ylim(0, 100)
    axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/argenziano_scores.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ График сохранён: /content/argenziano_scores.png')


def show_argenziano_examples(metrics, val_df, n=6):
    """
    Показывает примеры изображений с рассчитанными баллами.
    """
    probs = metrics['probs']
    scores_arr = np.array([compute_argenziano_score(probs[i])[0]
                           for i in range(len(probs))])

    # Берём по 3 из "подозрительных" и "нормы"
    susp_idx = np.where(scores_arr >= SUSPICION_THRESHOLD)[0]
    norm_idx  = np.where(scores_arr  < SUSPICION_THRESHOLD)[0]
    np.random.seed(42)
    chosen = np.concatenate([
        np.random.choice(susp_idx, min(3, len(susp_idx)), replace=False),
        np.random.choice(norm_idx,  min(3, len(norm_idx)),  replace=False),
    ])

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Примеры: расчёт баллов по шкале Argenziano', fontsize=13)

    for ax, idx in zip(axes.flat, chosen):
        score, verdict, found = compute_argenziano_score(probs[idx])
        img = Image.open(val_df.iloc[idx]['full_path']).convert('RGB')
        ax.imshow(img)
        ax.axis('off')

        weights_str = ' + '.join(
            [f'{ARGENZIANO_WEIGHTS[FEATURES[i]]}б ({FEAT_RU[i]})'
             for i, p in enumerate((probs[idx] >= THRESHOLD).astype(int))
             if p == 1]
        ) or 'нет признаков'

        color  = '#CC0000' if verdict == 1 else '#005500'
        status = '⚠ ПОДОЗРЕНИЕ' if verdict == 1 else '✓ НОРМА'
        title  = f'{status}  |  Балл: {score}\n{weights_str[:60]}'
        ax.set_title(title, fontsize=8, color=color, pad=4)

    plt.tight_layout()
    plt.savefig('/content/argenziano_examples.png', dpi=150, bbox_inches='tight')
    plt.show()


# Запускаем для всех экспериментов

print('\n' + '▓'*55)
print('  Подсчёт баллов Argenziano по всем экспериментам')
print('▓'*55)

scores2, verdicts2, gt2 = evaluate_argenziano(metrics2, 'Эксп.2 (ISIC 3ch)')
scores3, verdicts3, gt3 = evaluate_argenziano(metrics3, 'Эксп.3 (ISIC 4ch+маска)')
scores4, verdicts4, gt4 = evaluate_argenziano(metrics4, 'Эксп.4 (Derm+ISIC 4ch+aug)')

# Сравнительный график
plot_argenziano_scores({
    'Эксп.2 (ISIC 3ch)':       scores2,
    'Эксп.3 (ISIC 4ch+маска)': scores3,
    'Эксп.4 (Derm+ISIC 4ch+aug)': scores4,
})

# Примеры для лучшей модели (Эксп.4)
show_argenziano_examples(metrics4, val_df)

# Сохраняем на Drive
import shutil
for fname in ['argenziano_scores.png', 'argenziano_examples.png']:
    src = f'/content/{fname}'
    if os.path.exists(src):
        shutil.copy(src, os.path.join('/content/drive/MyDrive/Диплом/models', fname))
        print(f'✓ Сохранено: {fname}')

### 14. Grad-CAM

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def get_gradcam(model, img_tensor, class_idx):
    """Grad-CAM для EfficientNet-B3."""
    model.eval()
    gradients, activations = [], []

    def save_gradient(grad):
        gradients.append(grad)

    # на последний conv блок
    target_layer = model.blocks[-1]
    handle_f = target_layer.register_forward_hook(
        lambda m, i, o: activations.append(o)
    )
    handle_b = target_layer.register_backward_hook(
        lambda m, gi, go: save_gradient(go[0])
    )

    img_tensor = img_tensor.unsqueeze(0).to(device)
    img_tensor.requires_grad = True
    output = model(img_tensor)

    model.zero_grad()
    output[0, class_idx].backward()

    handle_f.remove()
    handle_b.remove()

    grad = gradients[0].mean(dim=[2, 3], keepdim=True)
    cam  = (grad * activations[0]).sum(dim=1).squeeze()
    cam  = torch.relu(cam).cpu().detach().numpy()
    cam  = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam

def show_gradcam_grid(model, df_sample, n_samples=4):
    """Показывает Grad-CAM для всех 7 признаков на n_samples изображениях."""
    samples = df_sample[df_sample[FEATURES].sum(axis=1) > 1].sample(n_samples, random_state=42)

    fig, axes = plt.subplots(n_samples, 8, figsize=(22, n_samples * 3))
    fig.suptitle('Grad-CAM: карты внимания модели (Эксп.4)', fontsize=13)

    for row, (_, sample) in enumerate(samples.iterrows()):
        # Загружаем изображение
        img = Image.open(sample['full_path']).convert('RGB')
        img_resized = img.resize((300, 300))
        img_arr = np.array(img_resized)

        # Тензор для модели (4 канала)
        ds_tmp = SkinDataset4Ch(pd.DataFrame([sample]), is_train=False)
        img_tensor, _ = ds_tmp[0]

        # Оригинал
        axes[row, 0].imshow(img_arr)
        diag = sample.get('diagnosis', '')
        axes[row, 0].set_title(f'Оригинал\n{diag[:15]}', fontsize=8)
        axes[row, 0].axis('off')

        # Grad-CAM для каждого признака
        for col, (feat, name) in enumerate(zip(FEATURES, FEAT_RU)):
            cam = get_gradcam(model4, img_tensor, col)
            cam_resized = np.array(
                Image.fromarray((cam * 255).astype(np.uint8)).resize((300, 300))
            ) / 255.

            axes[row, col+1].imshow(img_arr)
            axes[row, col+1].imshow(cam_resized, cmap='jet', alpha=0.45)
            gt  = int(sample[feat]) if sample[feat] >= 0 else -1
            axes[row, col+1].set_title(
                f'{name}\nGT={gt}', fontsize=7,
                color='red' if gt == 1 else 'gray'
            )
            axes[row, col+1].axis('off')

    plt.tight_layout()
    plt.savefig('/content/gradcam_exp4.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Grad-CAM сохранён: /content/gradcam_exp4.png')

# Запускаем на val выборке (Derm7pt)
show_gradcam_grid(model4, derm_val, n_samples=4)

### 15. Сохраняем модели и результаты на Google Drive

In [ ]:
import shutil
save_dir = '/content/drive/MyDrive/Диплом/models'
os.makedirs(save_dir, exist_ok=True)

files_to_save = [
    'best_exp2.pth', 'best_exp3.pth', 'best_exp4.pth',
    'results_summary.json',
    'eda_derm7pt.png', 'eda_isic.png',
    'all_experiments_comparison.png',
    'best_exp2_curves.png', 'best_exp3_curves.png', 'best_exp4_curves.png',
    'confusion_Эксп.2_ISIC_3ch.png',
    'confusion_Эксп.3_ISIC_4ch.png',
    'confusion_Эксп.4_Derm+ISIC_4ch.png',
]

for fname in files_to_save:
    src = f'/content/{fname}'
    if os.path.exists(src):
        shutil.copy(src, os.path.join(save_dir, fname))
        print(f'✓ {fname}')
    else:
        print(f'✗ не найдено: {fname}')
print('\nГотово! Все файлы сохранены на Google Drive.')
